# 01 · Очистка и подготовка данных CRM

**Проект:** анализ CRM онлайн-школы программирования X (июль 2023 – июнь 2024).
**Цель ноутбука:** превратить 4 «сырые» выгрузки CRM в чистые, связанные между собой таблицы,
на которых можно считать воронку, эффективность маркетинга и юнит-экономику.

| Таблица | Что содержит | Ключ |
|---|---|---|
| Contacts | контакты (лиды) | `Id` |
| Calls | звонки менеджеров | `CONTACTID` → Contacts |
| Spend | рекламные расходы по дням | `Source`, `Campaign` → Deals |
| Deals | потенциальные сделки | `Contact Name` → Contacts |

Вся логика очистки вынесена в модуль [`src/cleaning.py`](../src/cleaning.py), чтобы её можно было
переиспользовать и тестировать. Здесь — проверка исходных данных, запуск ETL и контроль результата.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
import cleaning as etl

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

## 1. Исходные данные

Берём `.xlsx`, а не `.csv`: в CSV-выгрузке кириллица в поле *Level of Deutsch* превратилась в `?`,
а 19-значные ID округлены как float (связь таблиц ломается). Все ID читаем как строки.

In [2]:
raw = etl.load_raw()
overview = pd.DataFrame({
    name: {"строк": len(df), "столбцов": df.shape[1],
           "полных дубликатов": df.duplicated().sum(),
           "ячеек с пропусками, %": round(df.isna().mean().mean() * 100, 1)}
    for name, df in raw.items()
}).T
overview

,строк,столбцов,полных дубликатов,"ячеек с пропусками, %"
Contacts,18548.0,4.0,0.0,0.0
Calls,95874.0,11.0,0.0,20.3
Spend,20779.0,8.0,917.0,11.8
Deals,21595.0,23.0,0.0,43.0


In [3]:
def missing_report(df):
    m = df.isna().sum()
    return (pd.DataFrame({"пропусков": m, "%": (m / len(df) * 100).round(1),
                          "тип": df.dtypes.astype(str)})
            .query("пропусков > 0").sort_values("%", ascending=False))

missing_report(raw["Deals"])

,пропусков,%,тип
Payment Type,21099,97.7,str
Months of study,20755,96.1,float64
Level of Deutsch,20344,94.2,object
City,19084,88.4,str
Education Type,18295,84.7,str
Course duration,18008,83.4,float64
Product,18003,83.4,str
Initial Amount Paid,17430,80.7,object
Offer Total Amount,17410,80.6,object
Term,9141,42.3,str


**Что видно в Deals до очистки:**
- `Payment Type`, `Months of study`, `Course duration`, `City`, `Level of Deutsch` заполнены у малой доли
  сделок — менеджер заполняет их только на поздних стадиях воронки. Это не ошибка, а особенность процесса:
  пропуск означает «до этого этапа сделка не дошла», поэтому такие поля **не удаляем и не заполняем средним**.
- Суммы хранятся текстом в разных форматах: `1000`, `€ 3.500,00`, `$11.000,00`.
- `SLA` — смесь `time` (чч:мм:сс) и `timedelta` (больше суток).

In [4]:
raw["Deals"][["Initial Amount Paid", "Offer Total Amount", "SLA", "Level of Deutsch", "City"]].dropna().sample(8, random_state=7)

,Initial Amount Paid,Offer Total Amount,SLA,Level of Deutsch,City
13959,1000,11000,02:09:27,б1,Oberhausen
9628,1000,5000,00:23:45,б1,Binzen
9548,1000,5000,01:00:18,в1,-
14504,1000,11000,10:33:54,б1,Troisdorf
14907,1000,11000,00:00:41,б1,Hagenow
3811,1000,5000,11:17:14,b1,-
11548,1000,5000,17:39:36,В1,Nürnberg
15005,1000,11500,01:21:18,А2 ( хочет просить совмещать),-


In [5]:
missing_report(raw["Calls"])

,пропусков,%,тип
Tag,95874,100.0,float64
Dialled Number,95874,100.0,float64
Outgoing Call Status,8999,9.4,str
Scheduled in CRM,8999,9.4,float64
CONTACTID,3933,4.1,str
Call Duration (in seconds),83,0.1,float64


## 2. Правила очистки и их обоснование

| # | Таблица | Проблема | Решение | Обоснование |
|---|---|---|---|---|
| 1 | Deals | `Lost Reason = Duplicate` | удалить | по FAQ это технический дубль лида; иначе завышаем число лидов и занижаем конверсию |
| 2 | Deals | `Closing Date` < `Created Time` | → `NaT` | закрыть сделку раньше создания невозможно (FAQ: ошибка, решает аналитик) |
| 3 | Deals | `Initial Amount Paid` > `Offer Total Amount` | 2 строки с offer 1 200 по 11-мес. курсу — опечатка в offer; остальные — initial ограничен offer | первый платёж не может превышать стоимость курса; консервативная оценка выручки |
| 4 | Deals | offer = 0 / 1 / 9 € | флаг `is_demo` | по FAQ это демо-доступы за символическую плату — не студенты |
| 5 | Deals | `Payment Done`, но `Months of study` пуст | не считаем студентом | FAQ: «если месяц не заполнен — студент не учится, деньги учитывать нельзя» |
| 6 | Deals | суммы текстом, SLA смешанного типа | → float / секунды | для расчётов нужны числа |
| 7 | Deals | `Level of Deutsch` свободным текстом (б1, в1, «B1 (до февраля)») | → CEFR `A0…C2` | кириллические б/в = B; берём текущий уровень (первый упомянутый) |
| 8 | Deals | города: `-`, адреса, разное написание | → `city`, `country` | для географического анализа |
| 9 | Deals | `Content`, `Term` | удалить | детализация объявления; кампания и источник сохраняются |
| 10 | Deals | категориальные пропуски | явные метки `Unknown`, `Not rated`, `Not lost` | пропуск сам по себе информативен; удаление строк исказило бы воронку |
| 11 | Calls | `Dialled Number`, `Tag` пустые на 100% | удалить | нет информации |
| 12 | Calls | нет длительности | удалить строки (83) | звонок без длительности нельзя анализировать |
| 13 | Calls | `Outgoing Call Status` пуст у входящих | → `Not applicable` | это не пропуск, а «не применимо» |
| 14 | Spend | 917 полных дублей | удалить | все — нулевые строки, на суммы не влияют |
| 15 | Spend | `Campaign` пуст | → `(no campaign)` | расход реальный, удалять нельзя |

**Выручка.** По FAQ факт оплаты — статус `Payment Done`, а сумма первого платежа — `Initial Amount Paid`.
Остаток курса оплачивается помесячно, поэтому выручка, полученная к дате выгрузки:

$$revenue = initial + (months\_of\_study - 1) \cdot \frac{offer - initial}{course\_duration - 1},\quad revenue \le offer$$

При единовременной оплате (`initial = offer`) выручка = `offer`.

In [6]:
clean, logs = etl.run_etl(save=True)
for name, lines in logs.items():
    print(f"\n=== {name}")
    for line in lines:
        print(" •", line)


=== Contacts
 • Дубликаты по id удалены: 0
 • created_time, modified_time -> datetime64 (формат дд.мм.гггг чч:мм)
 • Пропусков нет; contact_owner_name -> category

=== Calls
 • Дубликаты по id удалены: 0
 • Удалены полностью пустые столбцы: ['dialled_number', 'tag']
 • Удалены звонки без длительности/с отрицательной длительностью: 83
 • outgoing_call_status: NaN -> 'Not applicable' (входящие/пропущенные); scheduled_in_crm -> bool
 • Звонков без contact_id: 3855 — оставлены для анализа нагрузки менеджеров, но не участвуют в связке со сделками
 • Добавлен флаг is_conversation (>= 30 сек): медиана звонка 8 сек, короткие звонки — недозвоны/автоответчики

=== Spend
 • Полные дубликаты удалены: 917 (все — нулевые строки без расходов)
 • campaign/adgroup/ad: NaN -> '(no campaign)'/'(not set)' — расход реальный, просто не размечен, удалять его нельзя
 • Отрицательных значений impressions/spend/clicks: 0

=== Deals
 • Удалены пустые строки/дубли id: 2
 • Удалены технические дубли лидов (Lost R

## 3. Контроль результата

In [7]:
pd.DataFrame({
    name: {"строк до": len(raw[name]), "строк после": len(df),
           "столбцов": df.shape[1], "полных дубликатов": df.duplicated().sum()}
    for name, df in clean.items()
}).T

,строк до,строк после,столбцов,полных дубликатов
Contacts,18548,18548,4,0
Calls,95874,95791,10,0
Spend,20779,19862,8,0
Deals,21595,19822,30,0


In [8]:
deals = clean["Deals"]
deals.dtypes.to_frame("dtype").T

,id,deal_owner_name,closing_date,quality,stage,lost_reason,page,campaign,source,payment_type,product,education_type,created_time,course_duration,months_of_study,initial_amount_paid,offer_total_amount,contact_id,city,deal_cycle_days,sla_sec,amount_anomaly_flag,is_demo,is_paid,is_student,revenue,payments_count,country,german_level,month
dtype,str,category,datetime64[us],category,category,category,str,str,category,category,category,category,datetime64[us],float64,float64,float64,float64,str,str,float64,float64,bool,bool,bool,bool,float64,float64,category,category,datetime64[us]


In [9]:
checks = {
    "id сделок уникальны": deals["id"].is_unique,
    "нет closing_date < created_time": not (deals["closing_date"] < deals["created_time"].dt.normalize()).any(),
    "initial <= offer везде": not (deals["initial_amount_paid"] > deals["offer_total_amount"]).any(),
    "у студентов есть months_of_study": deals.loc[deals["is_student"], "months_of_study"].notna().all(),
    "у студентов есть продукт": deals.loc[deals["is_student"], "product"].ne("Unknown").all(),
    "выручка только у студентов": (deals.loc[~deals["is_student"], "revenue"] == 0).all(),
    "contact_id сделок есть в Contacts, %": round(deals["contact_id"].isin(clean["Contacts"]["id"]).mean() * 100, 1),
    "contact_id звонков есть в Contacts, %": round(clean["Calls"]["contact_id"].dropna().isin(clean["Contacts"]["id"]).mean() * 100, 1),
}
pd.Series(checks, name="проверка").to_frame()

,проверка
id сделок уникальны,True
нет closing_date < created_time,True
initial <= offer везде,True
у студентов есть months_of_study,True
у студентов есть продукт,True
выручка только у студентов,True
"contact_id сделок есть в Contacts, %",99.8
"contact_id звонков есть в Contacts, %",100.0


In [10]:
deals["german_level"].value_counts().to_frame("сделок")

,сделок
german_level,
Unknown,18623
B1,815
B2,169
A2,149
C1,27
A1,26
A0,10
C2,3


## 4. Итог

- Сделок после очистки: **19 822** (удалено 1 771 технический дубль), звонков: **95 791**, строк расходов: **19 862**.
- Студентов (оплата + заполнен срок обучения, не демо): **837** из 856 сделок `Payment Done`.
- Связь таблиц работает: 99.7% сделок и 100% звонков с контактом находят свой контакт.
- Очищенные таблицы сохранены в `data/processed/` в двух форматах: **CSV** (для Tableau/Sheets) и
  **pickle** (для Python: сохраняет типы `datetime`, `category`, `bool` и загружается мгновенно).

➡️ Следующий шаг — [02_eda_analysis.ipynb](02_eda_analysis.ipynb)